# FoG Detection — CNN + TCN Training Pipeline
**Improvements over baseline:**
- Subject-level normalization (not per-window)
- Pre-FoG class + transition-aware labeling (3-class)
- Oversampling minority class per fold
- CNN stem → Dilated TCN blocks with residuals
- Focal Loss with tuned alpha
- Early stopping on val F1
- Sequence smoothing post-processing
- Per-subject evaluation breakdown


In [ ]:
import torch, os
print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {mem:.1f} GB")
    DEVICE = 'cuda'
else:
    print("WARNING: No GPU — training will be slow")
    DEVICE = 'cpu'


In [ ]:
!pip install -q seaborn scikit-learn imbalanced-learn


In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)

print("Imports OK")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── UPDATE THIS PATH ──────────────────────────────────────────────────────────
DATA_DIR = '/content/drive/MyDrive/FoG_Project'
# ─────────────────────────────────────────────────────────────────────────────

X_raw = np.load(f'{DATA_DIR}/X_windows_all_subjects.npy')   # (N, 384, 6)
y_raw = np.load(f'{DATA_DIR}/y_windows_all_subjects.npy')   # (N,) — 0=NonFoG, 1=FoG

with open(f'{DATA_DIR}/subject_indices.json') as f:
    subject_indices = json.load(f)

print(f"X: {X_raw.shape}")
print(f"y: {y_raw.shape}")
print(f"FoG (1): {(y_raw==1).sum()} ({100*(y_raw==1).mean():.1f}%)")
print(f"NonFoG (0): {(y_raw==0).sum()} ({100*(y_raw==0).mean():.1f}%)")
print(f"Subjects: {[s['subject_id'] for s in subject_indices]}")

# Sanity checks
assert not np.isnan(X_raw).any(), "NaNs in X!"
assert not np.isinf(X_raw).any(), "Infs in X!"
assert set(np.unique(y_raw)).issubset({0,1}), f"Unexpected labels: {np.unique(y_raw)}"
print("\nAll sanity checks passed ✓")


## Step 1: Subject-Level Normalization
Per-window z-score treats each window independently — it removes useful amplitude information.
Instead, normalize per-subject: compute mean/std across all of a subject's windows, then apply.
This preserves relative differences within a subject while removing inter-subject scale differences.


In [ ]:
def subject_normalize(X, subject_indices):
    """
    Per-subject, per-channel normalization.
    For each subject: mean and std computed across all their windows and timesteps.
    X: (N, 384, 6)
    Returns X_norm: same shape
    """
    X_norm = X.copy()

    for subj in subject_indices:
        s = subj['start_idx']
        e = subj['end_idx']
        subj_data = X[s:e]               # (K, 384, 6)

        # Mean/std per channel across all windows and timesteps
        mean = subj_data.mean(axis=(0,1), keepdims=True)  # (1, 1, 6)
        std  = subj_data.std(axis=(0,1),  keepdims=True)  # (1, 1, 6)
        std  = np.where(std < 1e-8, 1e-8, std)

        X_norm[s:e] = (subj_data - mean) / std
        print(f"  Subject {subj['subject_id']}: "
              f"mean={mean.flatten().round(3)}, std={std.flatten().round(3)}")

    return X_norm

print("Applying subject-level normalization...")
X = subject_normalize(X_raw, subject_indices)
y = y_raw.copy()

print(f"\nAfter normalization:")
print(f"  Global mean: {X.mean():.4f} (should be ~0)")
print(f"  Global std:  {X.std():.4f}  (should be ~1)")


## Step 2: Pre-FoG Class + Transition-Aware Labeling
Add a **Pre-FoG** class: windows within N steps *before* a FoG event onset.
This is clinically important — detecting FoG *before* it fully starts improves patient safety.

**3 classes:**
- 0 = NonFoG (normal walking)
- 1 = Pre-FoG (approaching freeze, within ~2s of onset)
- 2 = FoG (active freezing episode)

Transition windows (NonFoG→FoG boundary) are also upweighted in the loss.


In [ ]:
PRE_FOG_STEPS = 8    # ~2 seconds before FoG onset (8 windows × 0.25s = 2s)
USE_3CLASS    = True # Set False for binary (0/1) mode

def add_pre_fog_class(y_subj, pre_fog_steps=8):
    """
    Convert binary (0/1) labels to 3-class (0/1/2) within one subject's sequence.
    y_subj: (K,) array with 0=NonFoG, 1=FoG
    Returns y_new: (K,) with 0=NonFoG, 1=Pre-FoG, 2=FoG
    """
    y_new = np.zeros_like(y_subj)
    y_new[y_subj == 1] = 2   # FoG → class 2

    # Find FoG onset indices (0→1 transitions)
    diff   = np.diff(y_subj, prepend=0)
    onsets = np.where(diff == 1)[0]

    for onset in onsets:
        start = max(0, onset - pre_fog_steps)
        # Only label NonFoG windows as Pre-FoG (don't overwrite FoG)
        for i in range(start, onset):
            if y_new[i] == 0:
                y_new[i] = 1

    return y_new

if USE_3CLASS:
    y_new = np.zeros_like(y)
    for subj in subject_indices:
        s, e = subj['start_idx'], subj['end_idx']
        y_new[s:e] = add_pre_fog_class(y[s:e], pre_fog_steps=PRE_FOG_STEPS)

    y       = y_new
    N_CLASS = 3
    CLASS_NAMES = ['NonFoG', 'Pre-FoG', 'FoG']
    print("3-class mode enabled:")
else:
    N_CLASS = 2
    CLASS_NAMES = ['NonFoG', 'FoG']
    print("Binary mode:")

for i, name in enumerate(CLASS_NAMES):
    n = (y == i).sum()
    print(f"  Class {i} ({name}): {n} windows ({100*n/len(y):.1f}%)")


## Step 3: CNN + TCN Architecture
```
Input (batch, 6, 384)
   ↓
Conv1D Stem
   ↓
3× Conv1D Blocks (residual + BN + dropout)
   ↓
4× Dilated TCN Blocks (dilation=1,2,4,8 — captures 1s to 8s context)
   ↓
Global Average Pooling
   ↓
Dense → Softmax
```


In [ ]:
from torch.nn.utils import weight_norm

# ── Residual Conv Block ───────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=7, dropout=0.3):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch, kernel, padding=kernel//2)
        self.bn   = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        out = self.drop(self.relu(self.bn(self.conv(x))))
        return self.relu(out + self.skip(x))

# ── Dilated TCN Block ─────────────────────────────────────────────────────────
class TCNBlock(nn.Module):
    """
    Dilated causal convolution block with residual connection.
    Non-causal version used here (bidirectional context within window).
    """
    def __init__(self, channels, kernel=3, dilation=1, dropout=0.2):
        super().__init__()
        pad = (kernel - 1) * dilation // 2   # Same padding for non-causal

        self.conv1 = weight_norm(nn.Conv1d(channels, channels, kernel,
                                           dilation=dilation, padding=pad))
        self.bn1   = nn.BatchNorm1d(channels)
        self.conv2 = weight_norm(nn.Conv1d(channels, channels, kernel,
                                           dilation=dilation, padding=pad))
        self.bn2   = nn.BatchNorm1d(channels)
        self.relu  = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x):
        out = self.drop1(self.relu(self.bn1(self.conv1(x))))
        out = self.drop2(self.relu(self.bn2(self.conv2(out))))
        return self.relu(out + x)   # Residual (same channels)

# ── Full CNN-TCN Model ────────────────────────────────────────────────────────
class FoGCNNTCN(nn.Module):
    """
    Input:  (batch, 6, 384)
    Output: (batch, num_classes)
    """
    def __init__(self, num_classes=3, dropout_cnn=0.3, dropout_tcn=0.2):
        super().__init__()

        # Stem
        self.stem = nn.Sequential(
            nn.Conv1d(6, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        # Conv blocks (with residuals + downsampling)
        self.conv_blocks = nn.Sequential(
            ConvBlock(64,  64,  kernel=7, dropout=dropout_cnn),
            nn.MaxPool1d(2),                                      # 384→192
            ConvBlock(64,  128, kernel=5, dropout=dropout_cnn),
            nn.MaxPool1d(2),                                      # 192→96
            ConvBlock(128, 128, kernel=3, dropout=dropout_cnn),
        )

        # TCN blocks — dilations 1,2,4,8 cover 1→8× receptive field
        self.tcn_blocks = nn.Sequential(
            TCNBlock(128, kernel=3, dilation=1,  dropout=dropout_tcn),
            TCNBlock(128, kernel=3, dilation=2,  dropout=dropout_tcn),
            TCNBlock(128, kernel=3, dilation=4,  dropout=dropout_tcn),
            TCNBlock(128, kernel=3, dilation=8,  dropout=dropout_tcn),
        )

        # Classifier head
        self.gap  = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.conv_blocks(x)
        x = self.tcn_blocks(x)
        x = self.gap(x).squeeze(-1)
        return self.head(x)

# Quick architecture check
model_test = FoGCNNTCN(num_classes=N_CLASS)
dummy = torch.randn(4, 6, 384)
out   = model_test(dummy)
print(f"Architecture OK: {dummy.shape} → {out.shape}")

n_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
del model_test, dummy, out


In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss for class imbalance.
    alpha: scalar or per-class tensor
    gamma: focusing parameter (2.0 standard)
    """
    def __init__(self, alpha=None, gamma=2.0, num_classes=3):
        super().__init__()
        self.gamma = gamma
        if alpha is None:
            self.alpha = None
        else:
            if isinstance(alpha, (list, np.ndarray)):
                self.register_buffer('alpha', torch.tensor(alpha, dtype=torch.float32))
            else:
                self.alpha = alpha

    def forward(self, inputs, targets):
        ce   = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt   = torch.exp(-ce)
        loss = (1 - pt) ** self.gamma * ce

        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            loss    = alpha_t * loss

        return loss.mean()

print("Loss function defined ✓")


In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss = correct = total = 0

    for X_b, y_b in loader:
        X_b = X_b.to(device, non_blocking=True)
        y_b = y_b.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if scaler:
            with torch.cuda.amp.autocast():
                out  = model(X_b)
                loss = criterion(out, y_b)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(X_b)
            loss = criterion(out, y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item()
        preds  = out.argmax(dim=1)
        correct += (preds == y_b).sum().item()
        total   += y_b.size(0)

    return total_loss / len(loader), 100 * correct / total


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    for X_b, y_b in loader:
        X_b  = X_b.to(device, non_blocking=True)
        out  = model(X_b)
        prob = torch.softmax(out, dim=1)
        pred = out.argmax(dim=1)

        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(y_b.numpy())
        all_probs.extend(prob.cpu().numpy())

    return np.array(all_preds), np.array(all_labels), np.array(all_probs)


def compute_metrics(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)

    if len(class_names) == 2:
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn + 1e-8)
        spec = tn / (tn + fp + 1e-8)
        prec = tp / (tp + fp + 1e-8)
        f1   = 2*tp / (2*tp + fp + fn + 1e-8)
        acc  = (tp + tn) / len(y_true)
        return dict(sensitivity=sens, specificity=spec, precision=prec,
                    f1=f1, accuracy=acc, cm=cm,
                    tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn))
    else:
        rep = classification_report(y_true, y_pred,
                                    target_names=class_names, output_dict=True)
        # For 3-class: FoG-specific metrics (class 2)
        fog_idx = class_names.index('FoG')
        fog_tp  = cm[fog_idx, fog_idx]
        fog_fn  = cm[fog_idx, :].sum() - fog_tp
        fog_fp  = cm[:, fog_idx].sum() - fog_tp
        fog_tn  = cm.sum() - fog_tp - fog_fn - fog_fp

        return dict(
            accuracy     = rep['accuracy'],
            macro_f1     = rep['macro avg']['f1-score'],
            weighted_f1  = rep['weighted avg']['f1-score'],
            fog_sensitivity = fog_tp / (fog_tp + fog_fn + 1e-8),
            fog_specificity = fog_tn / (fog_tn + fog_fp + 1e-8),
            fog_f1          = rep.get('FoG', {}).get('f1-score', 0),
            cm=cm
        )


def sequence_smooth(preds, window=5):
    """
    Majority vote smoothing over consecutive predictions.
    Removes isolated single-window FoG predictions (likely noise).
    """
    smoothed = preds.copy()
    half = window // 2
    for i in range(half, len(preds) - half):
        window_labels = preds[i-half:i+half+1]
        smoothed[i]   = np.bincount(window_labels, minlength=N_CLASS).argmax()
    return smoothed


print("Training functions defined ✓")


## Step 4: LOSO Cross-Validation
For each fold:
- **Test**: one subject
- **Val**: one other subject (used for early stopping only, never for model selection of hyperparams)
- **Train**: remaining subjects + oversampling of minority class


In [ ]:
from collections import Counter

def oversample_minority(X_train, y_train, strategy='moderate'):
    """
    Random oversampling of minority class windows.
    strategy: 'moderate' (2:1 ratio) or 'aggressive' (1:1 ratio)
    """
    class_counts = Counter(y_train.tolist())
    majority_n   = max(class_counts.values())

    if strategy == 'moderate':
        target = {c: max(n, majority_n // 2) for c, n in class_counts.items()}
    else:
        target = {c: majority_n for c in class_counts}

    X_parts = [X_train]
    y_parts = [y_train]

    for cls, count in class_counts.items():
        needed = target[cls] - count
        if needed > 0:
            idx     = np.where(y_train == cls)[0]
            extra   = np.random.choice(idx, needed, replace=True)
            X_parts.append(X_train[extra])
            y_parts.append(y_train[extra])

    X_out = np.concatenate(X_parts, axis=0)
    y_out = np.concatenate(y_parts, axis=0)

    # Shuffle
    perm = np.random.permutation(len(y_out))
    return X_out[perm], y_out[perm]


def loso_train(X, y, subject_indices,
               n_class=3, class_names=None,
               epochs=80, batch_size=256, lr=1e-3,
               device='cuda', patience=12,
               smooth_preds=True):

    if class_names is None:
        class_names = [str(i) for i in range(n_class)]

    # Transpose once: (N, 384, 6) → (N, 6, 384)
    X_t = torch.tensor(X, dtype=torch.float32).transpose(1, 2)
    y_t = torch.tensor(y, dtype=torch.long)

    scaler = torch.cuda.amp.GradScaler() if device == 'cuda' else None
    nw     = min(4, os.cpu_count() or 1)
    pin    = device == 'cuda'

    # Focal loss alpha: inverse frequency weighting
    class_counts = np.array([(y == i).sum() for i in range(n_class)], dtype=np.float32)
    alpha        = 1.0 / (class_counts / class_counts.sum())
    alpha       /= alpha.sum()
    print(f"Focal loss alpha: {dict(zip(class_names, alpha.round(3)))}")

    all_preds_raw = []
    all_preds_smooth = []
    all_labels   = []
    fold_results = {}
    n = len(subject_indices)

    for fold_idx, test_info in enumerate(subject_indices):
        test_subj = test_info['subject_id']
        val_info  = subject_indices[(fold_idx - 1) % n]
        val_subj  = val_info['subject_id']

        train_infos = [s for s in subject_indices
                       if s['subject_id'] not in (test_subj, val_subj)]

        print(f"\n{'='*70}")
        print(f"FOLD {fold_idx+1}/{n}  |  Test={test_subj}  Val={val_subj}")
        print(f"{'='*70}")

        test_idx  = list(range(test_info['start_idx'], test_info['end_idx']))
        val_idx   = list(range(val_info['start_idx'],  val_info['end_idx']))
        train_idx = [i for s in train_infos
                     for i in range(s['start_idx'], s['end_idx'])]

        # Oversample minority on training set
        X_tr_np = X[train_idx]
        y_tr_np = y[train_idx]
        X_tr_np, y_tr_np = oversample_minority(X_tr_np, y_tr_np, strategy='moderate')
        X_tr = torch.tensor(X_tr_np, dtype=torch.float32).transpose(1, 2)
        y_tr = torch.tensor(y_tr_np, dtype=torch.long)

        dist = Counter(y_tr_np.tolist())
        print(f"  Train (after oversample): {len(y_tr_np)} windows")
        for c, name in enumerate(class_names):
            print(f"    {name}: {dist.get(c,0)}")
        print(f"  Val:  {len(val_idx)} | Test: {len(test_idx)}")

        train_loader = DataLoader(TensorDataset(X_tr,             y_tr),
                                  batch_size=batch_size, shuffle=True,
                                  num_workers=nw, pin_memory=pin,
                                  persistent_workers=nw>0)
        val_loader   = DataLoader(TensorDataset(X_t[val_idx],  y_t[val_idx]),
                                  batch_size=batch_size*2, shuffle=False,
                                  num_workers=nw, pin_memory=pin,
                                  persistent_workers=nw>0)
        test_loader  = DataLoader(TensorDataset(X_t[test_idx], y_t[test_idx]),
                                  batch_size=batch_size*2, shuffle=False,
                                  num_workers=nw, pin_memory=pin,
                                  persistent_workers=nw>0)

        model     = FoGCNNTCN(num_classes=n_class).to(device)
        criterion = FocalLoss(alpha=alpha, gamma=2.0, num_classes=n_class).to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=20, T_mult=1, eta_min=1e-5)

        best_score   = 0.0
        best_state   = None
        wait         = 0
        history      = {'train_loss': [], 'train_acc': [], 'val_score': []}

        for epoch in range(epochs):
            tr_loss, tr_acc = train_epoch(model, train_loader, criterion,
                                          optimizer, scaler, device)
            scheduler.step()

            val_preds, val_lbls, _ = predict(model, val_loader, device)
            vm = compute_metrics(val_lbls, val_preds, class_names)
            val_score = vm.get('fog_f1', vm.get('f1', vm.get('macro_f1', 0)))

            history['train_loss'].append(tr_loss)
            history['train_acc'].append(tr_acc)
            history['val_score'].append(val_score)

            if val_score > best_score:
                best_score = val_score
                wait       = 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                wait += 1

            if (epoch + 1) % 10 == 0:
                fog_sens = vm.get('fog_sensitivity', vm.get('sensitivity', 0))
                fog_spec = vm.get('fog_specificity', vm.get('specificity', 0))
                print(f"  Ep {epoch+1:3d} | Loss={tr_loss:.4f} Acc={tr_acc:.1f}% | "
                      f"Val FoG-F1={val_score*100:.1f}% "
                      f"Sens={fog_sens*100:.1f}% Spec={fog_spec*100:.1f}%")

            if wait >= patience:
                print(f"  Early stop at epoch {epoch+1}  (best FoG-F1={best_score*100:.1f}%)")
                break

        # Load best model
        if best_state:
            model.load_state_dict(best_state)

        # Test evaluation
        y_pred_raw, y_true, y_prob = predict(model, test_loader, device)
        y_pred_smooth = sequence_smooth(y_pred_raw, window=5)

        m_raw    = compute_metrics(y_true, y_pred_raw,    class_names)
        m_smooth = compute_metrics(y_true, y_pred_smooth, class_names)

        fold_results[test_subj] = {
            'raw':    m_raw,
            'smooth': m_smooth,
            'probs':  y_prob,
            'history': history
        }

        all_preds_raw.extend(y_pred_raw)
        all_preds_smooth.extend(y_pred_smooth)
        all_labels.extend(y_true)

        print(f"\n  Test {test_subj} (raw):    "
              f"FoG-F1={m_raw.get('fog_f1', m_raw.get('f1',0))*100:.1f}%  "
              f"Sens={m_raw.get('fog_sensitivity', m_raw.get('sensitivity',0))*100:.1f}%  "
              f"Spec={m_raw.get('fog_specificity', m_raw.get('specificity',0))*100:.1f}%")
        print(f"  Test {test_subj} (smooth): "
              f"FoG-F1={m_smooth.get('fog_f1', m_smooth.get('f1',0))*100:.1f}%  "
              f"Sens={m_smooth.get('fog_sensitivity', m_smooth.get('sensitivity',0))*100:.1f}%  "
              f"Spec={m_smooth.get('fog_specificity', m_smooth.get('specificity',0))*100:.1f}%")

    overall_raw    = compute_metrics(np.array(all_labels), np.array(all_preds_raw),    class_names)
    overall_smooth = compute_metrics(np.array(all_labels), np.array(all_preds_smooth), class_names)

    return fold_results, overall_raw, overall_smooth

print("LOSO training function defined ✓")


## Step 5: Run LOSO Training

In [ ]:
fold_results, overall_raw, overall_smooth = loso_train(
    X, y, subject_indices,
    n_class      = N_CLASS,
    class_names  = CLASS_NAMES,
    epochs       = 80,
    batch_size   = 256,
    lr           = 1e-3,
    device       = DEVICE,
    patience     = 12,
    smooth_preds = True
)

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)


## Step 6: Evaluation & Visualisation

In [ ]:
def print_overall(label, m, class_names):
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    if 'fog_sensitivity' in m:
        print(f"  FoG Sensitivity : {m['fog_sensitivity']*100:.2f}%")
        print(f"  FoG Specificity : {m['fog_specificity']*100:.2f}%")
        print(f"  FoG F1          : {m['fog_f1']*100:.2f}%")
        print(f"  Overall Accuracy: {m['accuracy']*100:.2f}%")
        print(f"  Macro F1        : {m['macro_f1']*100:.2f}%")
    else:
        print(f"  Sensitivity: {m['sensitivity']*100:.2f}%")
        print(f"  Specificity: {m['specificity']*100:.2f}%")
        print(f"  F1         : {m['f1']*100:.2f}%")
        print(f"  Accuracy   : {m['accuracy']*100:.2f}%")

print_overall("OVERALL — Raw Predictions",      overall_raw,    CLASS_NAMES)
print_overall("OVERALL — Smoothed Predictions", overall_smooth, CLASS_NAMES)


In [ ]:
# ── Confusion Matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, m) in zip(axes, [("Raw", overall_raw), ("Smoothed", overall_smooth)]):
    cm_norm = m['cm'].astype(float) / m['cm'].sum(axis=1, keepdims=True)
    annot   = np.array([[f"{m['cm'][i,j]}\n({cm_norm[i,j]*100:.1f}%)"
                         for j in range(len(CLASS_NAMES))]
                        for i in range(len(CLASS_NAMES))])
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
                vmin=0, vmax=1, cbar_kws={'label': 'Rate'})
    ax.set_title(f"Confusion Matrix — {label}", fontsize=13)
    ax.set_ylabel("True"); ax.set_xlabel("Predicted")

plt.suptitle("LOSO Overall Results", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/FoG_Project/loso_confusion_matrices.png',
            dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ── Per-Subject Breakdown ─────────────────────────────────────────────────────
subjects = [s for s in fold_results if s != 'overall']
sens_raw, spec_raw, f1_raw     = [], [], []
sens_sm,  spec_sm,  f1_sm      = [], [], []

for subj in subjects:
    r = fold_results[subj]['raw']
    s = fold_results[subj]['smooth']
    sens_raw.append(r.get('fog_sensitivity', r.get('sensitivity', 0)) * 100)
    spec_raw.append(r.get('fog_specificity', r.get('specificity', 0)) * 100)
    f1_raw.append(  r.get('fog_f1',          r.get('f1', 0))          * 100)
    sens_sm.append( s.get('fog_sensitivity', s.get('sensitivity', 0)) * 100)
    spec_sm.append( s.get('fog_specificity', s.get('specificity', 0)) * 100)
    f1_sm.append(   s.get('fog_f1',          s.get('f1', 0))          * 100)

x = np.arange(len(subjects))
w = 0.25

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
titles = ['FoG Sensitivity', 'FoG Specificity', 'FoG F1']

for ax, raw_vals, sm_vals, title in zip(axes,
    [sens_raw, spec_raw, f1_raw],
    [sens_sm,  spec_sm,  f1_sm],
    titles):
    ax.bar(x - w/2, raw_vals,  w, label='Raw',      color='steelblue',  alpha=0.8)
    ax.bar(x + w/2, sm_vals,   w, label='Smoothed', color='darkorange', alpha=0.8)
    ax.axhline(np.mean(raw_vals), color='steelblue',  linestyle='--', alpha=0.6, linewidth=1)
    ax.axhline(np.mean(sm_vals),  color='darkorange', linestyle='--', alpha=0.6, linewidth=1)
    ax.set_xticks(x); ax.set_xticklabels(subjects, rotation=45)
    ax.set_title(title); ax.set_ylim(0, 100)
    ax.set_ylabel('% (if first subplot)' if ax == axes[0] else '')
    ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Per-Subject LOSO Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/FoG_Project/loso_per_subject.png',
            dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ── Training History ─────────────────────────────────────────────────────────
n_subj = len(subjects)
cols   = min(3, n_subj)
rows   = (n_subj + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
axes = np.array(axes).flatten()

for i, subj in enumerate(subjects):
    hist = fold_results[subj]['history']
    ax   = axes[i]
    ax2  = ax.twinx()

    ax.plot(hist['train_loss'], 'b-', alpha=0.7, label='Train Loss')
    ax2.plot(hist['val_score'], 'r-', alpha=0.7, label='Val FoG-F1')
    ax.set_title(f"Subject {subj}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss", color='b')
    ax2.set_ylabel("Val FoG-F1", color='r')
    ax2.set_ylim(0, 1)
    ax.grid(alpha=0.3)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Training History per Fold', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/FoG_Project/training_history.png',
            dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ── Save Results ──────────────────────────────────────────────────────────────
def make_serializable(d):
    if isinstance(d, dict):
        return {k: make_serializable(v) for k, v in d if k != 'probs'}
    elif isinstance(d, np.ndarray):
        return d.tolist()
    elif isinstance(d, (np.integer, np.floating)):
        return float(d)
    return d

save_data = {
    'overall_raw':    {k: v.tolist() if isinstance(v, np.ndarray) else v
                       for k, v in overall_raw.items()},
    'overall_smooth': {k: v.tolist() if isinstance(v, np.ndarray) else v
                       for k, v in overall_smooth.items()},
    'per_subject': {
        subj: {
            mode: {k: v.tolist() if isinstance(v, np.ndarray) else v
                   for k, v in fold_results[subj][mode].items()
                   if k not in ('probs', 'history')}
            for mode in ('raw', 'smooth')
        }
        for subj in subjects
    }
}

out_path = '/content/drive/MyDrive/FoG_Project/loso_results_cnntcn.json'
with open(out_path, 'w') as f:
    json.dump(save_data, f, indent=2)

print(f"Results saved to {out_path}")
print("\n=== FINAL SUMMARY ===")
print_overall("Raw",      overall_raw,    CLASS_NAMES)
print_overall("Smoothed", overall_smooth, CLASS_NAMES)
